# HomeBot Qwen3.5-4B Fine-tune (Unsloth, bf16 LoRA)

End-to-end notebook that takes the merged HomeBot multi-turn ChatML dataset,
fine-tunes **Qwen3.5-4B** with **bf16 LoRA** via Unsloth on a free Colab T4,
and exports a Q4_K_M GGUF that slots directly into the existing Ollama-based
DeepAgent runtime.

**Current dataset (run on 2026-04-18):**
- `qwen3_5_training.jsonl` -- 220 rows (204 synthetic + 16 real Telegram)
- `qwen3_5_val.jsonl` -- 24 rows (23 synthetic + 1 real Telegram)
- Quality gates applied: entity rename/drop, repair-prompt splicing,
  `<thinking>` stripping, min-3-word final assistant response, tool-call
  JSON validation, >=5x source-ratio warning, 5-sample spot-check.

Because the synthetic:real ratio is ~13:1, **step 5.5 oversamples the real
Telegram rows** before training so the model doesn't drift toward the
simulator's verbose style.

**Why bf16 LoRA (and not QLoRA 4-bit)?**
Unsloth explicitly warns that 4-bit QLoRA on Qwen3.5 produces *"higher than
normal quantization differences"*. bf16 LoRA needs ~2x more VRAM (10 GB vs
~6 GB) but fits T4 (16 GB) with room to spare and gives noticeably better
accuracy -- which matters for tool-calling since we need structured argument
output to parse cleanly at inference time.

**Training loop shape.** Each training example is a full multi-turn chain:

```
system -> user -> assistant+tool_calls -> tool -> ... -> assistant (final text)
```

We use `train_on_responses_only` so loss only applies to assistant +
tool_call spans -- the model is NOT rewarded for regurgitating the (long)
homebot system prompt.

Runtime target: ~20-30 minutes on free Colab T4 for ~2 epochs.

**Inputs required.**
1. A T4 (or better) Colab GPU runtime.
2. An HF **write** token pasted into step 0 below.

That's it -- `Runtime -> Run all` then go grab a coffee.


## 0. Configuration -- set your HF token once

Paste your Hugging Face **write** token in the cell below. Everything else
(dataset pull, optional model push) will reuse it, so the whole notebook can
be **Runtime -> Run all** without any interactive stops.

Token lives here so you don't need Colab Secrets or an `.env` file. Just
remember to clear it before sharing the notebook.

In [ ]:
import os

# --- REQUIRED: paste your HF write token here -----------------------------
HF_TOKEN = ""  # e.g. "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
# --------------------------------------------------------------------------

# --- Dataset + (optional) model repo names --------------------------------
HUB_DATASET_REPO = "kanakjr/homebot-qwen3.5"
HUB_MODEL_REPO   = "kanakjr/homebot-qwen3.5-4b-gguf"

# --- Knobs you rarely touch -----------------------------------------------
MODEL_NAME     = "unsloth/Qwen3.5-4B"
MAX_SEQ_LENGTH = 4096
REAL_OVERSAMPLE = 4       # step 5.5: how many times each real Telegram row is repeated
LORA_R = 16; LORA_ALPHA = 16  # bump both to 32 if under-fitting

# --- Propagate to env so downstream libs (datasets, huggingface_hub) see it
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

# Best-effort Colab Secrets fallback (only used if you left HF_TOKEN blank).
if not HF_TOKEN:
    try:
        from google.colab import userdata
        _secret = userdata.get("HF_TOKEN") or ""
        if _secret:
            HF_TOKEN = _secret
            os.environ["HF_TOKEN"] = _secret
            os.environ["HUGGING_FACE_HUB_TOKEN"] = _secret
            print("[config] using HF_TOKEN from Colab Secrets")
    except Exception:
        pass

assert HF_TOKEN, (
    "HF_TOKEN is empty -- paste your token in the cell above, or add it as a "
    "Colab Secret named HF_TOKEN."
)
print(f"[config] HF_TOKEN set (len={len(HF_TOKEN)})")
print(f"[config] dataset repo = {HUB_DATASET_REPO}")
print(f"[config] model   repo = {HUB_MODEL_REPO}")
print(f"[config] base model   = {MODEL_NAME}")

## 1. Install Unsloth + dependencies

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv

# Colab ships torchcodec pre-built against its bundled torch; when we pin
# torch==2.8.0+cu128 below, torchcodec's libtorchcodec_core*.so fails to
# load with "undefined symbol: _ZN3c104cuda..." and transitively breaks
# `from unsloth import FastLanguageModel`. We don't use torchcodec for LLM
# fine-tuning, so uninstall it unconditionally. Same story with torchao
# when its cuda kernels drift out of sync with the torch build.
!pip uninstall -y -qqq torchcodec torchao || true

if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil = f"pillow=={PIL.__version__}"
    except Exception:
        _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

# After the big install, re-uninstall torchcodec in case something pulled
# it back in as a transitive dep. Safe to skip if not present.
!pip uninstall -y -qqq torchcodec || true

## 2. Load Qwen3.5-4B in bf16 LoRA mode

Unsloth picks 16-bit LoRA automatically when you pass `load_in_4bit=False, load_in_16bit=True, full_finetuning=False`. Expect a log line saying *'Switching to 16bit LoRA'* -- that is the desired state for Qwen3.5 per Unsloth's guidance.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Auto-select the 16-bit dtype based on GPU capability:
#   T4 (Turing sm_75)   -> fp16 only. bf16 silently fails with
#                          "mat1 and mat2 to have the same dtype"
#                          when the LoRA adapters come out as fp16.
#   A100/L4/H100/etc.   -> bf16 (wider dynamic range, better for LLMs).
USE_BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)}  bf16 supported: {USE_BF16}  dtype: {DTYPE}")

# MODEL_NAME and MAX_SEQ_LENGTH come from step 0 (Configuration).
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = False,
    full_finetuning = False,
)


## 3. Attach LoRA adapters

`r=16, lora_alpha=16` is Unsloth's recommended Qwen3.5 starting point. If you see under-fitting after 2 epochs, bump both to 32.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


## 4. Apply the qwen3-instruct chat template

`qwen3-instruct` expects ChatML (`<|im_start|>role\ncontent<|im_end|>`). Our training JSONL already has `role=system/user/assistant/tool` with OpenAI-style `tool_calls`, so `apply_chat_template` handles everything. Thinking mode is **disabled by default** on small Qwen3.5 models which is exactly what we want for deterministic tool-calling.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)


## 5. Load the HomeBot dataset

Pulls from the private HF dataset repo set in step 0 (`HUB_DATASET_REPO`)
using the token from step 0. Set `USE_HUB = False` in the cell below if you
want to upload `qwen3_5_training.jsonl` + `qwen3_5_val.jsonl` from your
laptop instead (e.g. iterating locally without a re-push).

In [ ]:
from collections import Counter
from datasets import load_dataset

# Default path: pull the already-merged multi-turn dataset from the HF Hub
# repo set in step 0. Flip USE_HUB = False if you want to upload the two
# JSONLs from your laptop instead (e.g. iterating locally without pushing).
USE_HUB = True
LOCAL_TRAIN = "qwen3_5_training.jsonl"
LOCAL_VAL = "qwen3_5_val.jsonl"

if USE_HUB:
    ds = load_dataset(HUB_DATASET_REPO, token=HF_TOKEN)
else:
    if not (os.path.exists(LOCAL_TRAIN) and os.path.exists(LOCAL_VAL)):
        try:
            from google.colab import files  # noqa: F401
            print("Upload the two JSONL files when the file picker appears below...")
            uploaded = files.upload()  # expects qwen3_5_training.jsonl + qwen3_5_val.jsonl
            assert LOCAL_TRAIN in uploaded and LOCAL_VAL in uploaded, (
                f"Expected both {LOCAL_TRAIN} and {LOCAL_VAL}; got {list(uploaded)}"
            )
        except ImportError:
            raise RuntimeError(
                "Not running in Colab. Set USE_HUB=True, or place the two JSONLs in the CWD."
            )
    ds = load_dataset(
        "json",
        data_files={"train": LOCAL_TRAIN, "validation": LOCAL_VAL},
    )

print(ds)
train_sources = Counter(r.get("source", "?") for r in ds["train"])
val_sources = Counter(r.get("source", "?") for r in ds["validation"])
print(f"train sources: {dict(train_sources)}")
print(f"val   sources: {dict(val_sources)}")
print("\nTrain example keys:", list(ds["train"][0].keys()))
print("First user turn:", next(
    (m["content"][:140] for m in ds["train"][0]["messages"] if m["role"] == "user"),
    "(no user turn found)"
))


## 5.5 Oversample real Telegram rows

The merge step flagged a ~13:1 synthetic:real ratio. Real Telegram chats are
the most authentic signal for how the user actually talks (short, elliptical,
often no greeting), so we duplicate them `REAL_OVERSAMPLE` times inside the
train split. This is a "poor-man's sample weights" -- simpler to reason about
than trainer-level weighting, and it works cleanly with `train_on_responses_only`.

Rule of thumb: set `REAL_OVERSAMPLE` so the effective ratio ends up ~3:1 or
tighter. With 204 synthetic / 16 real in train, `REAL_OVERSAMPLE=4` gives
64 real, i.e. ~3.2:1.

In [ ]:
from datasets import concatenate_datasets

# REAL_OVERSAMPLE comes from step 0 (Configuration). Default 4.
real_rows = ds["train"].filter(lambda r: r.get("source") == "telegram")
synth_rows = ds["train"].filter(lambda r: r.get("source") != "telegram")

if len(real_rows) == 0:
    print("[oversample] no real rows found; skipping")
    train_balanced = ds["train"]
else:
    real_repeated = concatenate_datasets([real_rows] * REAL_OVERSAMPLE)
    train_balanced = concatenate_datasets([synth_rows, real_repeated]).shuffle(seed=3407)
    print(
        f"[oversample] real rows {len(real_rows)} -> {len(real_repeated)} "
        f"({REAL_OVERSAMPLE}x); synth unchanged at {len(synth_rows)}; "
        f"final train size = {len(train_balanced)}"
    )

ds_train = train_balanced
ds_val = ds["validation"]
print(f"train size={len(ds_train)}, val size={len(ds_val)}")

## 6. Render each conversation through the chat template

`apply_chat_template` with a ChatML template natively supports the `{role: tool, tool_call_id, name, content}` + `{role: assistant, tool_calls: [...]}` message shapes. We emit a single `text` field per example that SFTTrainer will tokenize.

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

# CRITICAL: drop every non-text column with `remove_columns=...`. If we leave
# `messages` in, SFTTrainer's default collator tries to tensorize the nested
# list-of-dicts and crashes with "Could not infer dtype of dict".
keep_only_text = lambda ds: ds.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=ds.column_names,
)

train_ds = keep_only_text(ds_train)
val_ds = ds_val
if val_ds is not None and len(val_ds) > 0:
    val_ds = keep_only_text(val_ds)

print(f"train_ds size={len(train_ds)}  val_ds size={0 if val_ds is None else len(val_ds)}")
print(f"train_ds columns after render: {train_ds.column_names}  (should be ['text'])")
print("\n--- Rendered example (truncated) ---\n")
print(train_ds[0]["text"][:2000])


## 7. Configure SFTTrainer

Small batch + high grad-accum fits Qwen3.5-4B bf16 LoRA on 16 GB T4. `max_seq_length=4096` covers the homebot system prompt + multi-turn conversation with some headroom.

In [ ]:
from trl import SFTTrainer, SFTConfig

# USE_BF16 was set in step 6 (model load). Mirror that choice to the trainer
# so autocast + grad-scaler pick the dtype that actually matches the weights.
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        max_length = 4096,
        packing = False,
        eval_strategy = "epoch" if val_ds is not None else "no",
        fp16 = not USE_BF16,
        bf16 = USE_BF16,
    ),
)


## 8. `train_on_responses_only` -- loss ONLY on assistant + tool_calls

Without this, the model learns to regurgitate the long system prompt, which is wasted capacity. Qwen3-instruct ChatML delimiters are:

- `<|im_start|>user\n` ... `<|im_end|>`
- `<|im_start|>assistant\n` ... `<|im_end|>`

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part   = "<|im_start|>assistant\n",
)


## 9. Memory snapshot before training

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved (before training).")


## 10. Train

In [ ]:
trainer_stats = trainer.train()


## 11. Memory snapshot after training

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_training = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
training_percentage = round(used_memory_training / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']:.1f} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_training} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage}%.")
print(f"Peak reserved memory for training % of max memory = {training_percentage}%.")


## 12. Sanity inference on held-out HomeBot prompts

These should emit a tool call (structured `tool_calls` array) when the right tool is obvious. Run 2-3 variations per skill family to eyeball that the fine-tune preserved tool-calling behavior.

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

# Each row tests a different skill family the dataset actually covered.
# "Healthy" output means a `<tool_call>{...}</tool_call>` block with the
# expected entity_id, followed by a short natural-language summary.
SANITY_PROMPTS = [
    # Home Assistant device control
    "turn off the air purifier",
    "dim the bedroom light to 40%",
    # Sensor query + synthesis rule
    "what's the temperature in the bedroom?",
    # Media pipeline
    "add the movie Dune Part Two",
    "search jellyseerr for succession",
    # Persistent memory
    "remember that my standing desk is switch.monitor_plug",
    "what did I say about deep work hours?",
    # Network admin
    "which devices are online right now?",
    # Obsidian / notes
    "summarize my notes from yesterday",
    # Interactive UI
    "what are my 3 favourite lights?",
]

# Matches the Modelfile fallback system prompt so behaviour at sanity-check
# time is close to what DeepAgent sends at runtime. At serve time the
# DeepAgent injects the canonical get_system_prompt() which supersedes this.
SYSTEM_PROMPT = (
    "You are HomeBotAI, an intelligent smart-home assistant powered by Home "
    "Assistant. The home is in India (IST timezone). Resident: Kanak.\n\n"
    "You have access to tools for Home Assistant control, media management "
    "(Sonarr/Radarr/Jellyfin/Jellyseerr), network admin (Deco mesh), "
    "Obsidian vault + persistent memory, link processing, interactive "
    "choices, and shell execution.\n\n"
    "Rules: be efficient with tool calls (1-3), summarize results in one "
    "short line, use colloquial names (never raw entity_ids), and stop "
    "after confirming an action."
)

for prompt in SANITY_PROMPTS:
    print("\n" + "=" * 60)
    print(f"USER: {prompt}")
    print("=" * 60)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    _ = model.generate(
        **inputs,
        max_new_tokens = 512,
        temperature = 0.7, top_p = 0.8, top_k = 20,
        streamer = TextStreamer(tokenizer, skip_prompt=True),
    )


## 13. Save LoRA adapters locally (checkpoint)

In [ ]:
model.save_pretrained("homebot-qwen3_5-lora")
tokenizer.save_pretrained("homebot-qwen3_5-lora")
print("Saved LoRA adapters to homebot-qwen3_5-lora/")


## 14. Export merged model to GGUF Q4_K_M for Ollama

This clones `llama.cpp`, merges the LoRA into a 16-bit model, and quantizes to Q4_K_M. Produces `homebot-qwen3_5.Q4_K_M.gguf` (~2.5 GB). Download via the Colab file browser, then on the Mac run:

```bash
ollama create homebot-qwen3_5 -f homebot_qwen3_5.Modelfile
```

In [ ]:
model.save_pretrained_gguf(
    "homebot-qwen3_5",
    tokenizer,
    quantization_method = "q4_k_m",
)
print("GGUF written to homebot-qwen3_5/ -- look for homebot-qwen3_5.Q4_K_M.gguf")


## 15. (Optional) Push GGUF + merged 16-bit weights to HF Hub

Flip the flags below if you want a reproducible uploaded artifact. Uses
`HF_TOKEN` and `HUB_MODEL_REPO` from step 0. Off by default -- the primary
delivery channel is a local Ollama import of the GGUF file from step 14.

In [ ]:
PUSH_GGUF_TO_HUB   = False
PUSH_MERGED_TO_HUB = False

# HUB_MODEL_REPO and HF_TOKEN both come from step 0.
if PUSH_GGUF_TO_HUB:
    model.push_to_hub_gguf(
        HUB_MODEL_REPO,
        tokenizer,
        quantization_method = ["q4_k_m"],
        token = HF_TOKEN,
    )
    print(f"Pushed GGUF to https://huggingface.co/{HUB_MODEL_REPO}")

if PUSH_MERGED_TO_HUB:
    model.push_to_hub_merged(
        HUB_MODEL_REPO + "-merged16",
        tokenizer,
        save_method = "merged_16bit",
        token = HF_TOKEN,
    )
    print(f"Pushed merged 16-bit weights to https://huggingface.co/{HUB_MODEL_REPO}-merged16")


## 16. Emit a ready-to-paste Ollama Modelfile

Copy the printed block into `homebot_qwen3_5.Modelfile` and run `ollama create homebot-qwen3_5 -f homebot_qwen3_5.Modelfile` on your Mac. The DeepAgent's `_resolve_model` already understands `ollama:homebot-qwen3_5`.

In [ ]:
# Keep this in sync with `finetuning/homebot_qwen3_5.Modelfile` in the repo.
# The template supports tool_call + tool role so Ollama routes structured
# tool_calls correctly. The SYSTEM block is only a FALLBACK -- at runtime the
# DeepAgent injects the canonical get_system_prompt() which supersedes this.
MODELFILE_TEMPLATE = r'''FROM ./homebot-qwen3_5.Q4_K_M.gguf

# Non-thinking Qwen3.5 sampling (matches Unsloth + Qwen docs).
PARAMETER temperature 0.7
PARAMETER top_p 0.8
PARAMETER top_k 20
PARAMETER repeat_penalty 1.05
PARAMETER num_ctx 8192
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
PARAMETER stop "<|endoftext|>"

TEMPLATE """{{- if .Messages }}
{{- range $i, $_ := .Messages }}
{{- if eq .Role "system" }}<|im_start|>system
{{ .Content }}<|im_end|>
{{ else if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
{{- if .Content }}
{{ .Content }}
{{- end }}
{{- if .ToolCalls }}
<tool_call>
{{ range .ToolCalls }}{"name": "{{ .Function.Name }}", "arguments": {{ .Function.Arguments }}}
{{ end }}</tool_call>
{{- end }}<|im_end|>
{{ else if eq .Role "tool" }}<|im_start|>user
<tool_response>
{{ .Content }}
</tool_response><|im_end|>
{{ end }}
{{- end }}<|im_start|>assistant
{{ end }}"""

SYSTEM """You are HomeBotAI, an intelligent smart-home assistant powered by Home Assistant.
The home is in India (IST timezone). Resident: Kanak.

You have access to tools for:
- Home Assistant device control (ha_call_service, ha_get_states, ha_search_entities)
- Media management (sonarr_*, radarr_*, jellyfin_*, jellyseerr_*, prowlarr_*, transmission_*)
- Network admin (deco_list_clients, deco_list_mesh_nodes, deco_reboot_nodes, deco_reservation_help)
- Obsidian vault + persistent memory (obsidian_*, memory_*)
- Link processing (process_and_save_link)
- Interactive choices (offer_choices -- tap-able buttons; end your turn after calling it)
- Shell execution (execute)

Rules:
1. Be efficient with tool calls -- 1-3 targeted calls over exhaustive searching.
2. Always provide a short natural-language summary after tool calls.
3. Use colloquial names in replies (e.g. "the purifier"), never raw entity_ids.
4. For short ordinal replies like "3" or "the second one", resolve against your
   previous message.
5. Confirm actions in one line and stop -- no filler tails, no second-guessing.
6. Synthesize redundant sensor readings (within ~1C / 5%RH) into one value instead
   of dumping raw lists.
"""
'''

MODELFILE_PATH = "homebot_qwen3_5.Modelfile"
with open(MODELFILE_PATH, "w") as f:
    f.write(MODELFILE_TEMPLATE)
print(f"Wrote {MODELFILE_PATH} ({len(MODELFILE_TEMPLATE)} bytes)")

print("\n=== Next steps (run on your Mac, NOT in Colab) ===")
print("1. Download both artifacts from the Colab file browser:")
print("      homebot-qwen3_5.Q4_K_M.gguf  (~2.5 GB)")
print(f"      {MODELFILE_PATH}")
print("2. Put them in the same directory on the Mac.")
print("3. ollama create homebot-qwen3_5 -f homebot_qwen3_5.Modelfile")
print("4. Quick sanity test:")
print("      ollama run homebot-qwen3_5 'turn off the air purifier'")
print("5. Point DeepAgent at the new model:")
print("      MODEL=ollama:homebot-qwen3_5")
print("   Then restart the DeepAgent server.")


---

**Troubleshooting**

- *`from unsloth import FastLanguageModel` fails with `OSError: libavutil.so.*` or `undefined symbol: _ZN3c104cuda29c10_cuda_check_implementationEiPKcS2_jb`*: Colab's pre-installed `torchcodec` is built against a different PyTorch build than the one the install cell pins. Fix in-place with `!pip uninstall -y torchcodec`, then re-run the import cell. **Runtime restart NOT required.** The updated step 1 cell already does this uninstall automatically, so this only affects sessions started before that fix.
- *`RuntimeError: expected mat1 and mat2 to have the same dtype, but got: c10::BFloat16 != c10::Half`*: you're on a T4 (Turing, sm_75) which does NOT have native bf16 support, but something (base weights or LoRA adapters) came out as bf16 anyway. Re-run the model-load cell -- it now auto-detects `USE_BF16` via `torch.cuda.is_bf16_supported()` and forces `dtype=torch.float16` on T4, then the SFTConfig cell picks `fp16=True, bf16=False` to match. If you hit this mid-training, re-run cells 3 (imports/model), 4 (LoRA attach), the dataset + oversample cells, and the trainer cell -- weights cannot be converted in-place without a reload.
- *OOM on T4*: drop `MODEL_NAME` to `unsloth/Qwen3.5-2B` and re-run. No other changes required.
- *Low eval loss but bad tool calls*: inspect the rendered example in step 6 -- make sure tool messages show up as `<|im_start|>tool ...`.
- *`transformers` version mismatch*: the install cell pins `transformers==5.2.0`; restart the runtime if you installed an earlier version in this session.
- *Tool calls fire but summary is verbose/robotic*: model is over-fitting the simulator style. Increase `REAL_OVERSAMPLE` in step 5.5 (try 6 or 8), or re-run `./run_pipeline.sh real --days 365 --limit 5000` locally to pull more genuine chats and re-merge.
- *Model only replies in natural language and never calls a tool*: you probably skipped step 8 (`train_on_responses_only`). Without it the model learns to reproduce the system prompt, crowding out tool-call patterns.
- *GGUF export fails with "unsupported architecture"*: Unsloth's `save_pretrained_gguf` needs the merged 16-bit model. Re-run step 14; if it still fails, bypass via `model.save_pretrained_merged(...)` and quantize manually with `llama.cpp/convert-hf-to-gguf.py`.

**Iteration recipe**

1. Download the GGUF + Modelfile, `ollama create`, try 5-10 real Telegram-style messages.
2. If tool arguments are wrong (wrong entity_id, missing field), that usually means the dataset is too small for that skill. Run `./run_pipeline.sh generate` to add more synthetic queries for that skill family, then `simulate` a handful and re-merge.
3. If the reply text style is off but tool calls are right, oversample real data more aggressively (step 5.5) rather than adding more synthetic rows.
